# Ablation Analysis (Python)

Python port of `ablation_analysis.ipynb`. Same three sections:

1. Decompose authored-helpful-notes by rater factor into (a) total authored and (b) fraction CRH. Plus a stacked status barplot.
2. Per-ablation-run summary vs baseline `coreRatingStatus`. Saved to `ablation_summary.tsv`.
3. Grid plot: % helpful retained by strategy x % dropped.

In [ ]:
import os

DATA_DIR  = "/home/jnallen/communitynotes/sourcecode/data_pre_june30"
RUNS_DIR  = "/home/jnallen/orcd/pool/communitynotes_data/ablation_runs/runs"
OUT_ROOT  = "/home/jnallen/orcd/pool/communitynotes_data/analysis"
PLOTS_DIR = f"{OUT_ROOT}/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams["figure.dpi"] = 100

## 1. Rater-factor decomposition

In [ ]:
rater_factors = (
    pd.read_parquet(f"{DATA_DIR}/helpfulness_scores.parquet",
                    columns=["raterParticipantId", "coreRaterFactor1"])
    .dropna(subset=["coreRaterFactor1"])
)
scored = pd.read_parquet(f"{DATA_DIR}/scored_notes.parquet",
                         columns=["noteId", "coreRatingStatus"])
notes_authors = pd.read_csv(f"{DATA_DIR}/notes-00000.tsv", sep="\t",
                            usecols=["noteId", "noteAuthorParticipantId"])

print(f"raters with factor: {len(rater_factors):,}")
print(f"scored notes:       {len(scored):,}")
print(f"notes with author:  {len(notes_authors):,}")

In [ ]:
# Per-author counts of authored notes by core scorer status.
joined = notes_authors.merge(scored, on="noteId", how="inner")
totals = joined.groupby("noteAuthorParticipantId").size().rename("n_total")
xt = pd.crosstab(joined["noteAuthorParticipantId"], joined["coreRatingStatus"])
xt = xt.rename(columns={
    "CURRENTLY_RATED_HELPFUL":     "n_crh",
    "CURRENTLY_RATED_NOT_HELPFUL": "n_crnh",
    "NEEDS_MORE_RATINGS":          "n_nmr",
})
for col in ("n_crh", "n_crnh", "n_nmr"):
    if col not in xt.columns:
        xt[col] = 0
counts = xt[["n_crh", "n_crnh", "n_nmr"]].join(totals).reset_index().rename(
    columns={"noteAuthorParticipantId": "raterParticipantId"})

per_rater = counts.merge(rater_factors, on="raterParticipantId", how="inner")
per_rater["frac_crh"] = per_rater["n_crh"] / per_rater["n_total"].clip(lower=1)
print(f"authors with factor + notes: {len(per_rater):,}")
per_rater.head()

In [ ]:
N_BINS = 15
brks = np.linspace(per_rater["coreRaterFactor1"].min(),
                   per_rater["coreRaterFactor1"].max(),
                   N_BINS + 1)
per_rater["bin"] = pd.cut(per_rater["coreRaterFactor1"], bins=brks, include_lowest=True)
per_rater["bin_mid"] = per_rater["bin"].cat.categories.map(lambda iv: iv.mid).take(
    per_rater["bin"].cat.codes).astype(float)

agg = (
    per_rater.groupby("bin_mid", observed=True)
    .agg(total_notes=("n_total", "sum"),
         total_crh=("n_crh", "sum"),
         total_crnh=("n_crnh", "sum"),
         total_nmr=("n_nmr", "sum"),
         n_raters=("n_total", "size"))
    .reset_index()
    .sort_values("bin_mid")
)
agg["frac_crh"] = agg["total_crh"] / agg["total_notes"]
agg

In [ ]:
bar_w = (brks[1] - brks[0]) * 0.95

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(agg["bin_mid"], agg["total_notes"], width=bar_w, color="steelblue")
ax1.set_xlabel("coreRaterFactor1 (bin midpoint)")
ax1.set_ylabel("total # notes authored")
ax1.set_title("Total notes written")

ax2.bar(agg["bin_mid"], agg["frac_crh"], width=bar_w, color="steelblue")
ax2.set_xlabel("coreRaterFactor1 (bin midpoint)")
ax2.set_ylabel("fraction CRH")
ax2.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax2.set_title("Fraction of authored notes rated helpful")

fig.suptitle("Decomposing authored helpful notes by rater factor")
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/rater_factor_decomp.png", dpi=200, bbox_inches="tight")
fig.savefig(f"{PLOTS_DIR}/rater_factor_decomp.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(agg))
ax.bar(agg["bin_mid"], agg["total_crnh"], width=bar_w, color="firebrick",
       label="Not helpful")
bottom = bottom + agg["total_crnh"].values
ax.bar(agg["bin_mid"], agg["total_nmr"], width=bar_w, bottom=bottom, color="#888888",
       label="Needs more ratings")
bottom = bottom + agg["total_nmr"].values
ax.bar(agg["bin_mid"], agg["total_crh"], width=bar_w, bottom=bottom, color="steelblue",
       label="Helpful")
ax.set_xlabel("coreRaterFactor1 (bin midpoint)")
ax.set_ylabel("# notes authored")
ax.set_title("Authored notes by core scorer status")
ax.legend()
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/rater_factor_status_stacked.png", dpi=200, bbox_inches="tight")
fig.savefig(f"{PLOTS_DIR}/rater_factor_status_stacked.pdf", bbox_inches="tight")
plt.show()

## 2. Per-ablation-run summary

Compares each run's `coreRatingStatus` against baseline `coreRatingStatus`. Saves `ablation_summary.tsv`.

In [ ]:
baseline_crh  = set(scored.loc[scored["coreRatingStatus"] == "CURRENTLY_RATED_HELPFUL",     "noteId"])
baseline_crnh = set(scored.loc[scored["coreRatingStatus"] == "CURRENTLY_RATED_NOT_HELPFUL", "noteId"])
baseline_nmr  = set(scored.loc[scored["coreRatingStatus"] == "NEEDS_MORE_RATINGS",          "noteId"])
print(f"baseline CRH:  {len(baseline_crh):,}")
print(f"baseline CRNH: {len(baseline_crnh):,}")
print(f"baseline NMR:  {len(baseline_nmr):,}")

In [ ]:
PATTERN = re.compile(r"^(extreme|central|random)_([0-9.]+)pct_seed([0-9]+)$")

def process_run(rd):
    m = PATTERN.match(rd.name)
    if not m:
        return None
    sn_path = rd / "scored_notes.tsv"
    if not sn_path.exists():
        return None
    sn = pd.read_csv(sn_path, sep="\t", usecols=["noteId", "coreRatingStatus"])
    run_crh  = set(sn.loc[sn["coreRatingStatus"] == "CURRENTLY_RATED_HELPFUL",     "noteId"])
    run_crnh = set(sn.loc[sn["coreRatingStatus"] == "CURRENTLY_RATED_NOT_HELPFUL", "noteId"])
    n_nmr    = int((sn["coreRatingStatus"] == "NEEDS_MORE_RATINGS").sum())
    return {
        "run":                rd.name,
        "strategy":           m.group(1),
        "pct":                float(m.group(2)),
        "seed":               int(m.group(3)),
        "n_crh":              len(run_crh),
        "n_crnh":             len(run_crnh),
        "n_nmr":              n_nmr,
        "n_new_crh":          len(run_crh  - baseline_crh),
        "n_dropped_crh":      len(baseline_crh  - run_crh),
        "n_new_crnh":         len(run_crnh - baseline_crnh),
        "n_dropped_crnh":     len(baseline_crnh - run_crnh),
        "pct_crh_recovered":  len(run_crh  & baseline_crh)  / max(len(baseline_crh),  1),
        "pct_crnh_recovered": len(run_crnh & baseline_crnh) / max(len(baseline_crnh), 1),
    }

rows = []
for rd in sorted(Path(RUNS_DIR).iterdir()):
    if not rd.is_dir():
        continue
    r = process_run(rd)
    if r is not None:
        rows.append(r)

summary_df = pd.DataFrame(rows)
out_path = f"{OUT_ROOT}/ablation_summary.tsv"
summary_df.to_csv(out_path, sep="\t", index=False)
print(f"wrote {len(summary_df)} rows to {out_path}")
summary_df

## 3. Grid plot: % helpful retained by strategy x % dropped

In [ ]:
grid = (
    summary_df.groupby(["strategy", "pct"])
    .agg(mean_retained=("pct_crh_recovered", "mean"),
         se_retained=("pct_crh_recovered",
                      lambda x: x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
         n_seeds=("pct_crh_recovered", "size"))
    .reset_index()
)
grid

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {"extreme": "#d62728", "central": "#2ca02c", "random": "#1f77b4"}
for strat, sub in grid.groupby("strategy"):
    sub = sub.sort_values("pct")
    ax.errorbar(sub["pct"], sub["mean_retained"], yerr=sub["se_retained"],
                marker="o", markersize=8, capsize=4,
                color=colors.get(strat, None), label=strat)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
ax.set_xlabel("% raters dropped")
ax.set_ylabel("% of baseline CRH notes retained")
ax.set_title("Ablation impact on helpful notes (core scorer)")
ax.legend(title="strategy")
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/ablation_grid_retained.png", dpi=200, bbox_inches="tight")
fig.savefig(f"{PLOTS_DIR}/ablation_grid_retained.pdf", bbox_inches="tight")
plt.show()